# Clustering Data To achieve ABC/XYZ classification 
#### Using GMM Model and K Mean clustering 

In [11]:
# unsupervised

import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.decomposition import PCA
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the dataset
df = pd.read_csv("Project IMS.csv")

# 1. Feature Selection
features = ['Month_1', 'Month_2', 'Month_3', 'Month_4', 'Month_5', 'Month_6', 'Month_7', 'Month_8', 'Month_9', 'Month_10', 'Month_11', 'Month_12', 'Annual_Consumption', 'Annual_Value', 'Mean_Demand', 'Std_Demand', 'CV', 'discountPercent', 'discountedSellingPrice']
X = df[features]
y_true = df['Combined_Class']

# 2. Pre-processing: Scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# --- Evaluation Metric: Purity Function ---
def calculate_purity(y_true, y_pred):
    # Calculates the Purity score for clustering.
    contingency_matrix = pd.crosstab(y_true, y_pred)
    return np.sum(np.max(contingency_matrix.values, axis=0)) / np.sum(contingency_matrix.values)

# --- 3. Perform Clustering (K=9) ---
n_clusters = 9

# K-Means
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init='auto')
df['kmeans_labels'] = kmeans.fit_predict(X_scaled)

# Gaussian Mixture Models (GMM)
gmm = GaussianMixture(n_components=n_clusters, random_state=42)
gmm.fit(X_scaled)
df['gmm_labels'] = gmm.predict(X_scaled)

# --- 4. Evaluate Metrics ---
# K-Means Evaluation
purity_kmeans = calculate_purity(y_true, df['kmeans_labels'])
ari_kmeans = adjusted_rand_score(y_true, df['kmeans_labels'])
nmi_kmeans = normalized_mutual_info_score(y_true, df['kmeans_labels'])

# GMM Evaluation
purity_gmm = calculate_purity(y_true, df['gmm_labels'])
ari_gmm = adjusted_rand_score(y_true, df['gmm_labels'])
nmi_gmm = normalized_mutual_info_score(y_true, df['gmm_labels'])

results = {
    'KMeans': {'Purity': purity_kmeans, 'ARI': ari_kmeans, 'NMI': nmi_kmeans},
    'GMM': {'Purity': purity_gmm, 'ARI': ari_gmm, 'NMI': nmi_gmm}
}
results_df = pd.DataFrame(results).T

# --- 5. Visualization (PCA, Heatmap, Scatter Plots) ---

# PCA for 2D Visualization
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
df['PCA1'] = X_pca[:, 0]
df['PCA2'] = X_pca[:, 1]

# K-Means Heatmap
kmeans_contingency = pd.crosstab(y_true, df['kmeans_labels'])
plt.figure(figsize=(10, 8))
sns.heatmap(kmeans_contingency, annot=True, fmt='d', cmap='Blues', linewidths=.5, linecolor='black')
plt.title('K-Means Clustering Contingency Matrix (Cluster vs Ground Truth)')
plt.ylabel('Ground Truth (Combined_Class)')
plt.xlabel('K-Means Cluster Label')
plt.savefig('kmeans_contingency_heatmap.png')
plt.close()

# GMM Heatmap
gmm_contingency = pd.crosstab(y_true, df['gmm_labels'])
plt.figure(figsize=(10, 8))
sns.heatmap(gmm_contingency, annot=True, fmt='d', cmap='Blues', linewidths=.5, linecolor='black')
plt.title('GMM Clustering Contingency Matrix (Cluster vs Ground Truth)')
plt.ylabel('Ground Truth (Combined_Class)')
plt.xlabel('GMM Cluster Label')
plt.savefig('gmm_contingency_heatmap.png')
plt.close()

# K-Means & Ground Truth Scatter Plot
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
sns.scatterplot(x='PCA1', y='PCA2', hue='kmeans_labels', data=df, palette='Spectral', legend='full', s=50)
plt.title('K-Means Clustering (K=9) on PCA Components')
plt.xlabel('Principal Component 1 (PCA1)')
plt.ylabel('Principal Component 2 (PCA2)')
plt.subplot(1, 2, 2)
sns.scatterplot(x='PCA1', y='PCA2', hue='Combined_Class', data=df, palette='Spectral', legend='full', s=50)
plt.title('Ground Truth Labels (Combined\_Class) on PCA Components')
plt.xlabel('Principal Component 1 (PCA1)')
plt.ylabel('Principal Component 2 (PCA2)')
plt.tight_layout()
plt.savefig('kmeans_groundtruth_comparison.png')
plt.close()

# GMM & Ground Truth Scatter Plot
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
sns.scatterplot(x='PCA1', y='PCA2', hue='gmm_labels', data=df, palette='Spectral', legend='full', s=50)
plt.title('GMM Clustering (K=9) on PCA Components')
plt.xlabel('Principal Component 1 (PCA1)')
plt.ylabel('Principal Component 2 (PCA2)')
plt.subplot(1, 2, 2)
sns.scatterplot(x='PCA1', y='PCA2', hue='Combined_Class', data=df, palette='Spectral', legend='full', s=50)
plt.title('Ground Truth Labels (Combined\_Class) on PCA Components')
plt.xlabel('Principal Component 1 (PCA1)')
plt.ylabel('Principal Component 2 (PCA2)')
plt.tight_layout()
plt.savefig('gmm_groundtruth_comparison.png')
plt.close()